In [2]:
# 데이터 생성
import pandas as pd

dataset = pd.read_csv('../data/csv/movie_reviews3.csv')

In [5]:
# 독립, 종속
X = dataset.iloc[ : , -1]
# 0.5 => 0으로, 3 => 1로, 5 => 2로
def convert(score):
	return 0 if score == 0.5 else 1 if score == 3 else 2
y = dataset['label'].apply(convert)


In [7]:
# 형태소 분석
from konlpy.tag import Okt
okt = Okt()

def tokenized_korean(text_list):
	return [" ".join(okt.morphs(text, stem=True)) for text in text_list]

morphs_korean = tokenized_korean(X)

In [8]:
# 벡터화
from tensorflow.keras import layers, models
vectorize_layer = layers.TextVectorization(
	max_tokens=1000,
	output_mode='int',
	output_sequence_length = 10
)

vectorize_layer.adapt(morphs_korean)

# layers.TextVectorization은 텍스트를 정수 시퀀스로 변환하는 도구입니다.
# max_tokens=1000: 전체 데이터에서 가장 자주 등장하는 단어 딱 1,000개만 기억하겠다는 뜻입니다. (너무 드물게 나오는 단어는 '알 수 없음(OOV)' 처리를 해서 모델의 혼란을 방지합니다.)
# output_mode='int': 각 단어를 고유한 **정수(ID)**로 매핑합니다. (예: "오늘" → 5, "학교" → 12)
# output_sequence_length=10: 모든 문장의 길이를 10개 단어로 맞춥니다.
# 단어가 10개보다 적으면? 뒤를 0으로 채웁니다(Padding).
# 단어가 10개보다 많으면? 뒤를 자릅니다(Truncating).


# 2. vectorize_layer.adapt(korean_morphs)의 역할
# 이 줄이 실제로 **학습(Fitting)**이 일어나는 부분입니다. korean_morphs 리스트를 훑으면서 다음과 같은 작업을 수행합니다.
# 빈도수 계산: 어떤 단어가 가장 많이 나왔는지 계산합니다.
# 단어 사전(Vocabulary) 생성: 1등 단어는 2번, 2등 단어는 3번... 이런 식으로 번호를 매깁니다. (0번은 패딩, 1번은 모르는 단어용으로 예약되는 경우가 많습니다.)


# 3. 결과물은 어떤 형태가 되나요?
# 만약 단어 사전이 `{"<OOV>": 1, "오늘": 2, "학교": 3, "가다": 4, ...}` 와 같이 만들어졌다면, 실제 문장은 다음과 같이 변합니다.
# 입력 데이터 (korean_morphs 중 하나):
# "오늘 학교 가다" (단어 3개)
# 벡터화 결과 (Vectorized Output):
# [2, 3, 4, 0, 0, 0, 0, 0, 0, 0]
# 앞의 2, 3, 4는 단어의 고유 번호입니다.
# 뒤의 0들은 문장 길이를 10으로 맞추기 위해 채워진 값입니다.

In [9]:
# 파이프라인
import tensorflow as tf
train_ds = tf.data.Dataset.from_tensor_slices((morphs_korean, y)).batch(8)

In [10]:
# 원문: "학교에 갔다."

# 1단계 (형태소 분석): ["학교", "에", "가다"] (단어별로 쪼개짐)

# 2단계 (벡터화): [12, 5, 8, 0, 0, ...] (단어가 숫자로 변신!)

# 3단계 (파이프라인): 위와 같은 숫자 묶음 8개를 박스에 담아 모델에게 전달.

In [ ]:
# 모델 설계
def build_positional_encoding():
	# 입력층
	inputs = layers.Input((1,), dtype=tf.string)

	#벡터화
	x = vectorize_layer(inputs)
	# 임베딩 : 단어를 
	vocab_size = vectorize_layer.vocabulary_size()
	word_embeding = layers.Embedding(input_dim=vocab_size, output_dim=64)(x)

	# 포지셔널 인코딩
	# 위치 정보 추가 : 0~9 번호를 생성
	positions = tf.range(start=0, limit=10, delta=1)
	# 임베딩 : 위치를 
	pos_embeding = layers.Embedding(input_dim=10, output_dim=64)(positions)

	# 의미 + 위치 
	x = word_embeding + pos_embeding

	# 멀티 헤드 어텐션(단어들끼리 서로 쳐다보며 "누가 누구랑 친한지" 관계도를 그립니다.)
	attention_output =layers.MultiHeadAttention(num_heads=2, key_dim=64)(x,x)

	# 잔차 계산 및 정규화
	x = layers.Add()([x, attention_output])
	x = layers.LayerNormalization()(x)

	# ffn
	ffn = layers.Dense(64, activation='relu')(x)
	x = layers.Add()([x, ffn])
	x = layers.LayerNormalization()(x)
	
	# 압축
	x = layers.GlobalAveragePooling1D()(x)
	#  출력층
	ouputs = layers.Dense(3, activation='softmax')(x)
	return models.Model(inputs=inputs, outputs=ouputs)

model = build_positional_encoding()

In [12]:
# 모델 설정
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [13]:
# 모델 학습
model.fit(train_ds, epochs=50, verbose=0)

In [23]:
import numpy as np
# 예측
# 테스트
# test_text = ["액션이 화려해요", "액션이 화려해서 재밌었어요", "액션이 화려하지만 지루해요"]
test_text = ['영화가 너무 재미있어서 하나도 안 지루해요', 
						 '영화가 너무 지루해서 하나도 안 재미있어요',
						 '인생 영화입니다',
						 '지루하고 재미없어요',
						 '영화가 너무 자극적이예요',
						 '배우연기가 별로예요']
# 형태소별로 문장을 수정
test_morphs = tokenized_korean(test_text)

# 테스트 할 때 텐서 타입을 변환(문자열 이슈)
test_input = tf.constant(test_morphs)

# 예측 실행
# 0 => 0.5, 1 => 3, 2=>5, 
predictions = model.predict(test_input)
predictions_size = len(predictions)

labels = [0.5, 3, 5]
# numpy의 argmax를 이용
# print(predictions[0]) # 첫번째 리뷰의 결과값
for i in range(predictions_size):
	idx = np.argmax(predictions[i])
	print(idx)
	print(predictions[i])
	p = predictions[i][idx] * 100
	print(f'{test_text[i]} : {labels[idx]}점({p:.2f}%)')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
2
[3.384671e-05 9.377768e-05 9.998723e-01]
영화가 너무 재미있어서 하나도 안 지루해요 : 5점(99.99%)
2
[3.081834e-05 9.022709e-05 9.998789e-01]
영화가 너무 지루해서 하나도 안 재미있어요 : 5점(99.99%)
2
[1.1206575e-06 1.0966400e-03 9.9890232e-01]
인생 영화입니다 : 5점(99.89%)
2
[1.0203794e-06 6.2269560e-04 9.9937624e-01]
지루하고 재미없어요 : 5점(99.94%)
2
[1.4551812e-04 1.8069122e-04 9.9967384e-01]
영화가 너무 자극적이예요 : 5점(99.97%)
1
[1.7972209e-04 8.7456024e-01 1.2526000e-01]
배우연기가 별로예요 : 3점(87.46%)
